# Seperate OAC runs
    Attempt to debug contrail output

In [1]:
import openairclim as oac
import xarray as xr
import matplotlib.pyplot as plt
from oac.utils.utility import load_species_inventory, generate_toml, time_norm_ncdf, scaled_emissions_to_nc

In [2]:
emi = xr.open_dataset('oac/repository/emi_inv_2025.nc')
print(emi)

<xarray.Dataset> Size: 27MB
Dimensions:   (index: 664350)
Coordinates:
  * index     (index) int64 5MB 0 1 2 3 4 ... 664345 664346 664347 664348 664349
Data variables:
    lon       (index) float32 3MB ...
    lat       (index) float32 3MB ...
    plev      (index) float32 3MB ...
    fuel      (index) float32 3MB ...
    CO2       (index) float32 3MB ...
    H2O       (index) float32 3MB ...
    NOx       (index) float32 3MB ...
    distance  (index) float32 3MB ...
Attributes:
    Title:           inventory_example
    Convention:      CF-XXX
    Inventory_Year:  2025
    Reference:       Based on DEPA 2050 data
    URL:             https://elib.dlr.de/142185/


In [3]:
print(emi.CO2)

<xarray.DataArray 'CO2' (index: 664350)> Size: 3MB
[664350 values with dtype=float32]
Coordinates:
  * index    (index) int64 5MB 0 1 2 3 4 ... 664345 664346 664347 664348 664349
Attributes:
    long_name:  CO2
    units:      kg


In [4]:
start_year = 1940
end_year = 2019
species_inventory_oac = load_species_inventory(start_year,end_year,"OAC")

## time norm with base being last year- 2019

In [5]:
inventory_file = f"mat_generated_nc_{2019}.nc"
scaled_emissions_to_nc("oac/repository/emi_inv_2025.nc", inventory_file,'Contrails',species_inventory_oac['Contrails'][-1],end_year)

Saved scaled emissions to oac_inputs\mat_generated_nc_2019.nc


In [6]:
species_list = list(species_inventory_oac.keys())

In [7]:
years = list(range(start_year, end_year + 1))
time_norm_ncdf(years , species_list, species_inventory_oac, nc_name="time_norm_debug.nc")

Saved new NetCDF to: time_evo/time_norm_debug.nc
Dataset summary:
<xarray.Dataset> Size: 960B
Dimensions:       (time: 80)
Coordinates:
  * time          (time) int32 320B 1940 1941 1942 1943 ... 2016 2017 2018 2019
Data variables:
    fuel          (time) float32 320B 11.15 12.05 13.01 ... 338.2 353.8 357.9
    dis_per_fuel  (time) float32 320B 0.06683 0.06683 0.06683 ... 0.1534 0.157
Attributes:
    Title:       Time normalization 
    Convention:  CF-XXX
    Type:        norm
    Author:      Abhigyan Prakash based on OAC example


In [8]:
ds = xr.open_dataset("time_evo/time_norm_debug.nc")

# Keep only 'fuel'
ds_fuel_only = ds[["fuel"]]

# Save it back or to a new file
ds_fuel_only.to_netcdf("time_evo/time_norm_fuel.nc")
print(xr.open_dataset("time_evo/time_norm_fuel.nc"))

<xarray.Dataset> Size: 640B
Dimensions:  (time: 80)
Coordinates:
  * time     (time) int32 320B 1940 1941 1942 1943 1944 ... 2016 2017 2018 2019
Data variables:
    fuel     (time) float32 320B ...
Attributes:
    Title:       Time normalization 
    Convention:  CF-XXX
    Type:        norm
    Author:      Abhigyan Prakash based on OAC example


In [9]:
generate_toml(start_year, end_year, step=1, output_file='debug_run.toml', specie_settings={}, 
                       inv_species=['distance'], out_species=['cont'], weighted=None, scaling= "norm", scale_file = "time_norm_historic_SSP.nc", inv_files=['mat_generated_nc_2019.nc'])

79
TOML file written to debug_run.toml


In [10]:
generate_toml(start_year, end_year, step=1, output_file='debug_run_fuel.toml', specie_settings={}, 
                       inv_species=['distance'], out_species=['cont'], weighted=None, scaling= "norm", scale_file = "time_norm_fuel.nc", inv_files=['mat_generated_nc_2019.nc'])

79
TOML file written to debug_run_fuel.toml


In [11]:
oac.run('tomls/debug_run.toml')

PermissionError: [WinError 32] Le processus ne peut pas accéder au fichier car ce fichier est utilisé par un autre processus: 'oac_results/gen_1940s.nc'

In [ ]:
oac.run('tomls/debug_run_fuel.toml')

In [ ]:
generate_toml(start_year, end_year, step=1, output_file='debug_run_ei.toml', specie_settings={}, 
                       inv_species=['distance'], out_species=['cont'], weighted=None, scaling= "norm", scale_file = "time_norm_debug.nc", inv_files=['mat_generated_nc_2019.nc'])

In [ ]:
oac.run('tomls/debug_run_ei.toml')

## Time norm with base being first year - 1940

In [ ]:
inventory_file = f"mat_generated_nc_{1940}.nc"
scaled_emissions_to_nc("oac/repository/emi_inv_2025.nc", inventory_file,'Contrails',species_inventory_oac['Contrails'][0],start_year)

In [ ]:
generate_toml(start_year, end_year, step=1, output_file='debug_run2.toml', specie_settings={}, 
                       inv_species=['distance'], out_species=['cont'], weighted=None, scaling= "norm", scale_file = "time_norm_historic_SSP.nc", inv_files=['mat_generated_nc_1940.nc'])

In [ ]:
generate_toml(start_year, end_year, step=1, output_file='debug_run_fuel2.toml', specie_settings={}, 
                       inv_species=['distance'], out_species=['cont'], weighted=None, scaling= "norm", scale_file = "time_norm_fuel.nc", inv_files=['mat_generated_nc_1940.nc'])

In [ ]:
oac.run('tomls/debug_run2.toml')

In [ ]:
oac.run('tomls/debug_run_fuel2.toml')

In [ ]:
generate_toml(start_year, end_year, step=1, output_file='debug_run_ei2.toml', specie_settings={}, 
                       inv_species=['distance'], out_species=['cont'], weighted=None, scaling= "norm", scale_file = "time_norm_debug.nc", inv_files=['mat_generated_nc_1940.nc'])

In [ ]:
oac.run("tomls/debug_run_ei2.toml")